# Nemesis — Backtest Analysis

Evaluates two strategies over historical J-Quants Pro data:

1. **MultiFactorStrategy** — Normal market: TDnet events + supply/demand + US overnight + macro
2. **ShockRecoveryStrategy** — After US market drops: buy oversold JP stocks with no bad news

**Trading assumption:** Buy at T+1 open, sell at T+2 open (overnight hold)

**No look-ahead bias:** Signals use only data available BEFORE market open

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != 'Nemesis' else os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from datetime import date
import pandas as pd
import numpy as np

print('✅ Setup complete')

In [ ]:
# ─── Backtest configuration ───────────────────────────────────────────
BT_START = date(2024, 1, 4)    # Start date
BT_END   = date(2024, 12, 30)  # End date
TOP_N    = 20                   # Portfolio size
SLIPPAGE_BPS = 5.0              # One-way slippage

print(f'Backtest: {BT_START} to {BT_END}, top_n={TOP_N}, slippage={SLIPPAGE_BPS}bps')

## Step 1: Load Historical Data

Fetches all daily quotes for the backtest period from J-Quants Pro.

In [ ]:
from japan_stock_daily.collectors.jquants_collector import JQuantsCollector
from datetime import timedelta

jquants = JQuantsCollector()

# Fetch all daily quotes for backtest period + 40 days buffer
# NOTE: This may take a few minutes for a 1-year range
print('Fetching historical quotes from J-Quants Pro...')
print('(This will take a few minutes for a 1-year range)')

all_rows = []
current = BT_START - timedelta(days=45)  # Buffer for 30-day lookback
end_with_buffer = BT_END + timedelta(days=5)

from tqdm.auto import tqdm
dates = []
d = current
while d <= end_with_buffer:
    if d.weekday() < 5:  # Weekdays only
        dates.append(d)
    d += timedelta(days=1)

for d in tqdm(dates, desc='Fetching daily quotes'):
    try:
        daily = jquants.get_daily_quotes(d)
        if daily is not None and not daily.empty:
            all_rows.append(daily)
    except Exception as e:
        pass  # Skip missing dates

all_quotes = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
print(f'\nLoaded {len(all_quotes):,} daily quote records across {all_quotes["code"].nunique():,} stocks')

In [ ]:
# Load supply/demand data for the period
print('Fetching margin and short selling data...')
margin_rows, short_rows, breakdown_rows = [], [], []

for d in tqdm(dates[::5], desc='Fetching supply/demand'):  # Every 5 days (weekly data)
    try:
        m = jquants.get_weekly_margin_interest(d)
        if m is not None and not m.empty:
            margin_rows.append(m)
    except Exception:
        pass
    try:
        s = jquants.get_short_selling_positions(d)
        if s is not None and not s.empty:
            short_rows.append(s)
    except Exception:
        pass

all_margin = pd.concat(margin_rows, ignore_index=True) if margin_rows else pd.DataFrame()
all_short  = pd.concat(short_rows, ignore_index=True) if short_rows else pd.DataFrame()
print(f'Margin records: {len(all_margin):,} | Short records: {len(all_short):,}')

## Step 2: Run Shock Recovery Backtest

In [ ]:
from backtest.engine import BacktestEngine
from backtest.strategies.shock_recovery import ShockRecoveryStrategy
from japan_stock_daily.collectors.us_overnight_collector import USOverNightCollector

# Build data loader that provides US overnight + supply/demand per date
us_collector = USOverNightCollector()
universe = jquants.get_listed_companies()

def data_loader(signal_date):
    us_data = us_collector.get_historical_overnight(signal_date)
    # Get latest margin/short data before signal_date
    margin_latest = all_margin[all_margin['date'].dt.date <= signal_date] if not all_margin.empty else pd.DataFrame()
    if not margin_latest.empty:
        margin_latest = margin_latest.sort_values('date').groupby('code').last().reset_index()
    short_latest = all_short[all_short['date'].dt.date <= signal_date] if not all_short.empty else pd.DataFrame()
    if not short_latest.empty:
        short_latest = short_latest.sort_values('date').groupby('code').last().reset_index()
    return {
        'us_overnight': us_data,
        'margin': margin_latest,
        'short': short_latest,
        'universe': universe,
        # TDnet/EDINET disabled in backtest for speed (re-enable for accuracy)
        'tdnet': pd.DataFrame(),
        'edinet': pd.DataFrame(),
    }

engine = BacktestEngine(all_quotes=all_quotes, slippage_bps=SLIPPAGE_BPS, data_loader=data_loader)

print('Running Shock Recovery Strategy backtest...')
shock_result = engine.run(
    ShockRecoveryStrategy(top_n=TOP_N, shock_threshold=0.03, volume_spike_min=1.8),
    start_date=BT_START,
    end_date=BT_END,
)
shock_result.print_summary()

## Step 3: Parameter Sweep — Optimize Shock Recovery

In [ ]:
print('Parameter sweep for ShockRecoveryStrategy (may take 10-15 minutes)...')

sweep_results = engine.sweep_parameters(
    strategy_class=ShockRecoveryStrategy,
    param_grid={
        'shock_threshold':  [0.02, 0.03, 0.04, 0.05],
        'volume_spike_min': [1.5, 2.0, 2.5, 3.0],
        'top_n':            [10, 20],
        'hold_days':        [1, 2, 3],
        'target_pct':       [0.02, 0.03, 0.04],
        'stop_pct':         [0.015, 0.02, 0.03],
    },
    start_date=BT_START,
    end_date=BT_END,
)

print('\nTop 10 parameter combinations by Sharpe Ratio:')
display_cols = ['shock_threshold', 'volume_spike_min', 'top_n',
                'sharpe_ratio', 'win_ratio', 'annual_return', 'max_drawdown', 'profit_factor']
sweep_results[[c for c in display_cols if c in sweep_results.columns]].head(10)

## Step 4: Equity Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Equity curve
ax1 = axes[0]
cum_shock = shock_result.get_cum_returns()
ax1.plot(cum_shock.index, cum_shock.values, label='Shock Recovery', color='#e94560', linewidth=2)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Cumulative Returns (Equity Curve)', fontsize=14)
ax1.set_ylabel('Portfolio Value (1.0 = Starting Capital)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Daily PnL distribution
ax2 = axes[1]
shock_pnl = shock_result.daily_pnl * 100
ax2.hist(shock_pnl[shock_pnl > 0], bins=30, alpha=0.6, color='green', label='Wins')
ax2.hist(shock_pnl[shock_pnl < 0], bins=30, alpha=0.6, color='red', label='Losses')
ax2.axvline(x=0, color='black', linewidth=1)
ax2.set_title('Daily Return Distribution (%)', fontsize=14)
ax2.set_xlabel('Daily Return (%)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/backtest_results.png')

## Step 5: Trade Log Analysis

In [ ]:
trade_log = shock_result.trade_log
print(f'Total trades: {len(trade_log)}')
print(f'Win ratio: {(trade_log["is_win"].sum() / len(trade_log) * 100):.1f}%')
print(f'Avg win: {trade_log[trade_log["is_win"]]["return_pct"].mean():.3f}%')
print(f'Avg loss: {trade_log[~trade_log["is_win"]]["return_pct"].mean():.3f}%')
print()

# Best trades
print('Top 10 Best Trades:')
trade_log.nlargest(10, 'return_pct')[[
    'signal_date', 'entry_date', 'code', 'entry_price', 'exit_price', 'return_pct'
]]


In [ ]:
# Monthly returns heatmap
from backtest.metrics import compute_monthly_returns

monthly = compute_monthly_returns(shock_result.daily_pnl)
monthly['monthly_pct'] = monthly['monthly_return'] * 100

print('Monthly Returns (%)')
print(monthly['monthly_pct'].round(2).to_string())

## Step 6: Calendar Analysis (Day-of-Week Effect)

In [ ]:
# Day-of-week performance
pnl_df = shock_result.daily_pnl.to_frame('return')
pnl_df.index = pd.to_datetime(pnl_df.index)
pnl_df['weekday'] = pnl_df.index.day_name()
pnl_df['return_pct'] = pnl_df['return'] * 100

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
day_stats = pnl_df.groupby('weekday')['return_pct'].agg(['mean', 'count', lambda x: (x > 0).mean() * 100])
day_stats.columns = ['avg_return', 'n_days', 'win_rate']
day_stats = day_stats.reindex([d for d in day_order if d in day_stats.index])

print('Day-of-Week Performance (Shock Recovery):')
day_stats

---
## Section 4: LightGBM Walk-Forward Optimisation

Replaces the hand-crafted `shock_threshold` / `volume_spike` rules with a **LightGBM** model.
Optuna tunes LightGBM hyperparameters on each walk-forward fold by maximising out-of-sample
**ranking Sharpe** (= Sharpe of the top-quartile predictions on the held-out period).

Protocol:
- Train window : 180 calendar days (≈9 months)
- Prediction window: 60 calendar days (≈3 months)
- Roll forward every 60 days → no look-ahead bias
- Best params re-fitted on full train window before each prediction fold

In [ ]:
# ── Install if needed ────────────────────────────────────────────────────────
# !pip install lightgbm optuna scikit-learn -q


In [ ]:
from backtest.feature_engineer import build_feature_matrix, FEATURE_COLS
from backtest.strategies.lgbm_shock_recovery import LGBMShockRecoveryStrategy
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

print("LightGBM version:", lgb.__version__)
print("Optuna version:  ", optuna.__version__)


### 4.1  Build Feature Matrix from Rule-Based Backtest Trade Log

We re-use the `rule_result.trade_log` from the rule-based run above as seed data.
Each trade row gets enriched with all raw features (price, volume, technical,
macro, disclosure flags) so LightGBM can learn from them.

In [ ]:
# Build feature matrix — enriches every trade in rule_result with raw features
# (Assumes `rule_result`, `all_quotes`, `topix_quotes`, `us_data_by_date`,
#  `tdnet_by_date`, `edinet_by_date`, `short_df`, `margin_df`, `universe_df`
#  are already loaded from earlier sections)

feature_matrix = build_feature_matrix(
    trade_log      = rule_result.trade_log,
    quotes         = all_quotes,
    topix_quotes   = topix_quotes,
    us_data_by_date= us_data_by_date,
    tdnet_by_date  = tdnet_by_date,
    edinet_by_date = edinet_by_date,
    short_df       = short_df,
    margin_df      = margin_df,
    universe_df    = universe_df,
)

print(f"Feature matrix: {len(feature_matrix):,} trades × {len(FEATURE_COLS)} features")
print(f"Win rate (>0.5%): {feature_matrix['target_win_50bps'].mean():.1%}")
print(f"Mean return:      {feature_matrix['target_return'].mean()*100:.3f}%")
feature_matrix[FEATURE_COLS].describe().T[['mean','std','min','max']].round(4)


### 4.2  Walk-Forward LightGBM Training

For each 60-day prediction window, Optuna searches over LightGBM hyperparameters
(learning_rate, num_leaves, max_depth, min_child_samples, subsample, colsample_bytree,
reg_alpha, reg_lambda, n_estimators) by maximising the **ranking Sharpe** on a
70/30 time-series split within the training window.

In [ ]:
lgbm_strategy = LGBMShockRecoveryStrategy(
    top_n          = 20,
    hold_days      = 3,
    target_pct     = 0.03,
    stop_pct       = 0.02,
    target_col     = "target_win_50bps",   # Binary: win probability
    n_optuna_trials= 50,                   # Increase for better tuning (e.g. 100)
    lgbm_seed      = 42,
)

print("Training walk-forward LightGBM models...")
lgbm_strategy.train(feature_matrix)
print(f"Folds fitted: {len(lgbm_strategy._models)}")
print("Best params per fold:")
for i, p in enumerate(lgbm_strategy._best_params_log):
    print(f"  Fold {i+1}: lr={p.get('learning_rate','-'):.4f}  "
          f"leaves={p.get('num_leaves','-')}  depth={p.get('max_depth','-')}  "
          f"n_est={p.get('n_estimators','-')}")


### 4.3  Feature Importance

In [ ]:
fi = lgbm_strategy.feature_importance()
if not fi.empty:
    top20 = fi.head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top20["feature"][::-1], top20["avg_importance"][::-1], color="steelblue")
    ax.set_xlabel("Average Gain Importance (across walk-forward folds)")
    ax.set_title("LightGBM Feature Importance — Shock Recovery")
    # Error bars for std
    ax.barh(top20["feature"][::-1], top20["std_importance"][::-1],
            left=top20["avg_importance"][::-1], alpha=0.3, color="red", label="±1 std")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print("\nTop 10 features:")
    print(fi.head(10).to_string(index=False))
else:
    print("No feature importance available (no models fitted)")


### 4.4  Run Backtest with LightGBM Strategy

In [ ]:
lgbm_result = engine.run(
    strategy   = lgbm_strategy,
    start_date = backtest_start,
    end_date   = backtest_end,
    progress   = True,
)
lgbm_result.print_summary()


### 4.5  Compare Rule-Based vs LightGBM

In [ ]:
from backtest.metrics import compute_metrics

# Build comparison table
comparison = pd.DataFrame({
    "Rule-based":  compute_metrics(rule_result.daily_pnl),
    "LightGBM":    compute_metrics(lgbm_result.daily_pnl),
}).T

key_cols = ["win_ratio", "sharpe_ratio", "sortino_ratio",
            "annual_return", "max_drawdown", "profit_factor", "n_trades"]
print("\n=== Rule-Based vs LightGBM Comparison ===")
print(comparison[key_cols].to_string())

# Cumulative return chart
fig, ax = plt.subplots(figsize=(12, 5))
rule_result.get_cum_returns().plot(ax=ax, label="Rule-Based ShockRecovery", color="blue")
lgbm_result.get_cum_returns().plot(ax=ax, label="LightGBM ShockRecovery",  color="darkorange")
ax.axhline(1.0, color="black", linewidth=0.5, linestyle="--")
ax.set_title("Cumulative Returns: Rule-Based vs LightGBM (Walk-Forward)")
ax.set_ylabel("Cumulative Return (1 = 100%)")
ax.legend()
plt.tight_layout()
plt.show()


### 4.6  Optuna Learning Curve

Visualise how Optuna improves the objective across trials for the last fold.

In [ ]:
# Re-run a single Optuna study with trial tracking (for visualisation only)
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_contour,
)

# Use last 30% of feature_matrix as the analysis fold
fm_sorted = feature_matrix.sort_values("_signal_date")
split_idx = int(len(fm_sorted) * 0.70)
train_viz = fm_sorted.iloc[:split_idx]
X_viz = train_viz[FEATURE_COLS].copy()
y_viz = train_viz["target_win_50bps"]
for col in ["sector_code","shock_is_macro","disc_has_negative","disc_has_positive"]:
    if col in X_viz.columns:
        X_viz[col] = X_viz[col].astype("category")

val_viz = fm_sorted.iloc[split_idx:]
X_val_viz = val_viz[FEATURE_COLS].copy()
y_val_viz = val_viz["target_win_50bps"]
for col in ["sector_code","shock_is_macro","disc_has_negative","disc_has_positive"]:
    if col in X_val_viz.columns:
        X_val_viz[col] = X_val_viz[col].astype("category")

from backtest.strategies.lgbm_shock_recovery import _ranking_sharpe

def viz_objective(trial):
    params = {
        "objective":         "binary",
        "metric":            "binary_logloss",
        "verbosity":         -1,
        "learning_rate":     trial.suggest_float("learning_rate",  0.01, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves",        16, 128),
        "max_depth":         trial.suggest_int("max_depth",          3,   8),
        "min_child_samples": trial.suggest_int("min_child_samples", 10,  60),
        "subsample":         trial.suggest_float("subsample",       0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree",0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha",      1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda",     1e-4, 10.0, log=True),
    }
    n_est = trial.suggest_int("n_estimators", 100, 500)
    try:
        dtrain = lgb.Dataset(X_viz, label=y_viz, categorical_feature=list(
            ["sector_code","shock_is_macro","disc_has_negative","disc_has_positive"]),
            free_raw_data=False)
        model = lgb.train(params, dtrain, num_boost_round=n_est)
        preds = model.predict(X_val_viz)
        score = _ranking_sharpe(preds, y_val_viz.values)
        return -score
    except Exception:
        return 0.0

viz_study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
viz_study.optimize(viz_objective, n_trials=80, show_progress_bar=True)

print(f"\nBest trial Sharpe: {-viz_study.best_value:.4f}")
print(f"Best params:")
for k, v in viz_study.best_params.items():
    print(f"  {k}: {v}")

# Plot optimisation history (requires plotly)
try:
    fig_hist  = plot_optimization_history(viz_study)
    fig_imp   = plot_param_importances(viz_study)
    fig_hist.show()
    fig_imp.show()
except Exception as e:
    print(f"Plotly visualisation skipped: {e}")


### 4.7  Exit Reason Analysis

How often does the adaptive exit (target hit vs stop-loss hit vs max-hold) trigger?

In [ ]:
for strategy_name, result in [("Rule-Based", rule_result), ("LightGBM", lgbm_result)]:
    tl = result.trade_log
    if "exit_reason" not in tl.columns:
        continue
    print(f"\n{strategy_name} — exit reason breakdown:")
    summary = (
        tl.groupby("exit_reason")["return_pct"]
        .agg(count="count", avg_ret="mean", win_rate=lambda x: (x>0).mean())
        .round(3)
    )
    print(summary.to_string())
